In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# RandomForestClassifierの仕様
04_baseline_modelに、RandomForestClassifierを使ったモデルを考える。また、データの分割やパラメータの調整によって性能が変わってくるため、クロスバリデーションを導入する。モデル部分以外は、04と同じ内容。

# データ読み込み

In [2]:
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

# 特徴量の追加

In [3]:
train_csv["Sex_Pclass"] = train_csv["Sex"] + "_" + train_csv["Pclass"].astype(str)

train_csv["logFare"] = np.log1p(train_csv["Fare"])

train_csv["Family"] = train_csv[["Parch","SibSp"]].sum(axis=1)
train_csv["len_Fam"] = train_csv["Family"].apply(lambda x:"alone" if x==0 else "basic" if 1<=x<=3 else "large")
train_csv=train_csv.drop("Family",axis=1)

# テストデータへの特徴量の追加

In [4]:
test_csv["Sex_Pclass"] = test_csv["Sex"] + "_" + test_csv["Pclass"].astype(str)

test_csv["logFare"] = np.log1p(test_csv["Fare"])

test_csv["Family"] = test_csv[["Parch","SibSp"]].sum(axis=1)
test_csv["len_Fam"] = test_csv["Family"].apply(lambda x:"alone" if x==0 else "basic" if 1<=x<=3 else "large")
test_csv=test_csv.drop("Family",axis=1)


# 特徴量の選定

In [5]:
import pandas as pd

#目的変数を分離
y = train_csv.Survived
X = train_csv.drop("Survived",axis=1)

#低カーディナリティの特徴量の選定(取り扱い安くノイズにならないもの)
cat_cols = [c for c in X.columns if X[c].nunique()<=10 and X[c].dtype == "object"]

#数値列の選定
num_cols = [c for c in X.columns if X[c].dtype in ["int64","float64"]]

#使用する特徴量の全体
using_cols = cat_cols + num_cols
X_train = X[using_cols].copy()

# 前処理法

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

#数値列への処理方法
num_transformer = SimpleImputer(strategy = "median")

#カテゴリー列への処理方法
cat_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy = "most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

#それぞれの前処理をまとめる
preprocessor = ColumnTransformer(
    transformers=[
        ("num",num_transformer,num_cols),
        ("cat",cat_transformer,cat_cols)
    ]
)

# モデルの選定

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=500,random_state=10,n_jobs=-1)

# cross validationの導入

In [8]:
from sklearn.model_selection import cross_val_score

my_pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",model)
])


scores = cross_val_score(my_pipeline,X_train,y,cv=5,scoring="accuracy")

#モデルに学習
#my_pipeline.fit(X_train,y)

#予測の作成
#preds = my_pipeline.predict(X_train)

#予測精度の測定
#score = accuracy_score(y_valid,preds)

print(scores.mean())

0.8036281463812692


In [9]:
#X、yの全体のデータを使って学習
my_pipeline.fit(X_train,y)

#全体データを学習したモデルで予測作成
preds = my_pipeline.predict(test_csv)

output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)